# 02 · Relacionar sin multiplicar y detectar episodios (35–47 min)

Una observación de temperatura por encima del límite no es un fallo mecánico confirmado. Cinco muestras consecutivas producen un episodio en esta regla didáctica.

In [ ]:
import json
import pandas as pd
from industrialdata.common import root
from industrialdata.catalog import load_catalog
from industrialdata.ingest import ingest, business_tables
folder = root()
catalog = load_catalog(folder)  # Consulta PostgreSQL: NO el catálogo JSON de pruebas.
from industrialdata.analyze import analyze
telemetry, _, _, _ = ingest(folder / "raw", catalog, "exploracion-mini")
production, maintenance = business_tables(folder / "raw", catalog)
enriched = telemetry.merge(pd.DataFrame(catalog["machines"]), on="machine_id", validate="many_to_one")
assert len(enriched) == len(telemetry) == 40
print("Filas antes/después del join:", len(telemetry), len(enriched))
print("Piezas correctas en la fuente:", production.units_ok.sum())

No sumes producción después de unirla a cada medición por máquina: repetirías las mismas piezas muchas veces. El soporte relaciona la producción con intervalos `[inicio, fin)` y calcula los agregados a una granularidad común.

In [ ]:
episodes, quality, metrics = analyze(telemetry, production, maintenance, catalog)
display(episodes)
assert len(episodes) == 1
assert episodes.iloc[0].start_utc == "2026-01-12T08:02:00Z"
assert episodes.iloc[0].end_utc == "2026-01-12T08:07:00Z"

In [ ]:
display(telemetry[telemetry.sensor_id == "M01_temperature"][["timestamp_utc", "value"]])
from industrialdata.reglas import superaciones, estados
print("79, 80 y 81 frente a 80:", superaciones([79, 80, 81], 80))

## Tabla de casos de A4: muestra, no episodio
Los siguientes casos ilustran las decisiones. La duración mínima de diez muestras se verifica después; no estamos afirmando que estos ejemplos aislados sean paradas reales.

In [ ]:
cases = pd.DataFrame({"produccion_cero_conocida": [True, True, True, False], "potencia_kw": [6, 0, 0, 0], "mantenimiento_planificado": [False, True, False, True]})
flags = estados(cases.produccion_cero_conocida.tolist(), cases.potencia_kw.tolist(), cases.mantenimiento_planificado.tolist(), .1)
display(cases.assign(**flags))

**Preguntas:** ¿coincidir con mantenimiento demuestra causalidad? No. ¿El hueco de M02 significa parada? No. Sus dos muestras ausentes afectan a calidad; no llegan a las diez necesarias para un episodio de sensor ausente en el trabajo original.